# Notebook 01: Data Audit and Exploratory Data Analysis (EDA)
**Author:** Mate (35%)

This notebook performs the initial data quality audit, descriptive statistics, and exploratory data analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data
We only load the 3 selected stations: Aotizhongxin, Changping, and Dingling.

In [ ]:
raw_dir = '../data/raw/PRSA_Data_20130301-20170228'
# If the directory is different due to extraction, we can also search recursively:
files = glob.glob('../data/raw/**/*.csv', recursive=True)
target_stations = ['Aotizhongxin', 'Changping', 'Dingling']

dfs = []
for f in files:
    if any(station in f for station in target_stations):
        df = pd.read_csv(f)
        dfs.append(df)

df_raw = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(df_raw)} rows from {len(dfs)} stations.")
df_raw.head()

## 2. Data Quality Audit

In [ ]:
# Missing values per column
missing_audit = df_raw.isnull().sum().to_frame(name='Missing Count')
missing_audit['Missing Percentage (%)'] = (missing_audit['Missing Count'] / len(df_raw)) * 100
print("Missing Values Audit:")
display(missing_audit)

# Duplicate rows
dupes = df_raw.duplicated().sum()
print(f"\nNumber of duplicate rows: {dupes}")

## 3. Descriptive Statistics

In [ ]:
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.drop(['No', 'year', 'month', 'day', 'hour'])
desc_stats = df_raw[numeric_cols].describe()
display(desc_stats)

## 4. Seasonal Trends of PM2.5
To answer our questions about winter inversions, we must see how PM2.5 fluctuates over the year.

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='month', y='PM2.5', hue='station', data=df_raw)
plt.title('Monthly PM2.5 Distribution by Station (2013-2017)')
plt.ylabel('PM2.5')
plt.xlabel('Month')
plt.legend(loc='upper right')
plt.savefig('../figures/monthly_pm25_boxplot.png')
plt.show()

## 5. Correlations among Co-pollutants
This supports the need for PCA later.

In [ ]:
pollutants = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3']
corr_matrix = df_raw[pollutants].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap of Pollutants')
plt.savefig('../figures/pollutant_correlation.png')
plt.show()